# EdgeGuard-Road semantic training stack probe

Execution-only wrapper for `EGX-SEG-STACK-*`. It uses synthetic inputs, requires CUDA, verifies five forward/backward paths and one exact checkpoint resume, and stops before real Cityscapes training. Timing and memory are framework feasibility evidence, not scientific performance.

In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

EDGEGUARD_REPOSITORY_URL = "https://github.com/emrealmaoglu/edgeguard-road.git"
EDGEGUARD_EXPECTED_COMMIT = "REPLACE_WITH_REVIEWED_EG_SEG_001_COMMIT_SHA"
PROJECT_ROOT = Path("/content/edgeguard-road")
MMSEG_CHECKOUT = Path("/content/edgeguard-mmseg")
OUTPUT_DIR = Path("/content/edgeguard-stack-probe")

if not re.fullmatch(r"[0-9a-f]{40}", EDGEGUARD_EXPECTED_COMMIT):
    raise ValueError("Set EDGEGUARD_EXPECTED_COMMIT to the exact reviewed 40-character SHA")
for target in (PROJECT_ROOT, MMSEG_CHECKOUT, OUTPUT_DIR):
    if target.exists():
        raise FileExistsError(f"Refusing existing runtime target: {target.name}")
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True

In [ ]:
subprocess.run(
    [
        "git",
        "clone",
        "--filter=blob:none",
        "--no-checkout",
        EDGEGUARD_REPOSITORY_URL,
        str(PROJECT_ROOT),
    ],
    check=True,
)
subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "checkout", "--detach", EDGEGUARD_EXPECTED_COMMIT], check=True
)
actual_commit = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
git_status = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "status", "--porcelain=v1"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if actual_commit != EDGEGUARD_EXPECTED_COMMIT or git_status:
    raise RuntimeError("Project checkout identity or cleanliness verification failed")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[dev]"], check=True)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime; CUDA is required for this stack probe")
print(
    {"torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0)}
)
subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / "scripts/train/install_semantic_stack.py"),
        "--config",
        str(PROJECT_ROOT / "configs/training/segmentation/framework_mmseg.yaml"),
        "--checkout",
        str(MMSEG_CHECKOUT),
        "--execute",
    ],
    check=True,
    cwd=PROJECT_ROOT,
)

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/unit/test_training_lab.py",
        "tests/integration/test_notebook.py",
    ],
    check=True,
    cwd=PROJECT_ROOT,
)
subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / "scripts/train/train_semantic.py"),
        "validate-configs",
        "--config-root",
        str(PROJECT_ROOT / "configs/training/segmentation"),
    ],
    check=True,
    cwd=PROJECT_ROOT,
)

In [ ]:
subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / "scripts/train/train_semantic.py"),
        "stack-probe",
        "--config-root",
        str(PROJECT_ROOT / "configs/training/segmentation"),
        "--mmseg-checkout",
        str(MMSEG_CHECKOUT),
        "--output-dir",
        str(OUTPUT_DIR),
        "--project-root",
        str(PROJECT_ROOT),
        "--project-commit",
        EDGEGUARD_EXPECTED_COMMIT,
    ],
    check=True,
    cwd=PROJECT_ROOT,
)

In [ ]:
import json

from google.colab import files

completion = json.loads((OUTPUT_DIR / "completion.json").read_text(encoding="utf-8"))
if completion["model_count"] != 5 or not completion["checkpoint_resume_verified"]:
    raise RuntimeError("Stack probe completion contract failed")
if completion["scientific_accuracy_evidence"] is not False:
    raise RuntimeError("Synthetic probe cannot be scientific evidence")
package = OUTPUT_DIR / completion["evidence_package"]
subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / "scripts/train/verify_semantic_training_artifact.py"),
        "--package",
        str(package),
        "--sha256",
        completion["evidence_package_sha256"],
    ],
    check=True,
    cwd=PROJECT_ROOT,
)
files.download(str(package))

## Stop gate

Stop here. Do not mount Drive or begin Cityscapes training. Human review must accept the exact framework evidence, real EG-DATA-002 preparation, and one split before EG-SEG-002.